# Training LLMs to Follow Instructions with Human Feedback: A Deep Dive into InstructGPT

This notebook provides an in-depth, hands-on exploration of the concepts presented in the OpenAI paper **"Training language models to follow instructions with human feedback"** (arXiv:2203.02155v1) by Ouyang et al. We will dissect and implement the core methodology used to create **InstructGPT**, a model that is significantly better at following user intentions than its base model, GPT-3.

The central idea is to align a powerful, pre-trained Large Language Model (LLM) with human intent using a three-step process involving **Reinforcement Learning from Human Feedback (RLHF)**.

## Section 1: Overview & Prerequisites

This section outlines the foundational knowledge required for this notebook and provides a roadmap for what we'll cover.

### 1.1 Summary of the Research

The paper demonstrates that simply increasing the size of language models does not inherently make them better at following a user's intent. Large models can still generate outputs that are untruthful, toxic, or unhelpful. The authors propose a technique to *align* LLMs with user intent on a wide range of tasks by fine-tuning with human feedback.

The process consists of three main steps:
1.  **Step 1: Supervised Fine-Tuning (SFT)**: Collect a dataset of human-written demonstrations of desired behavior on various prompts and use it to fine-tune a pre-trained GPT-3 model.
2.  **Step 2: Reward Modeling (RM)**: Collect a dataset of human-labeled comparisons, where humans rank several model outputs for a given prompt. This data is used to train a reward model that learns to predict which outputs humans would prefer.
3.  **Step 3: Reinforcement Learning (RL)**: Use the reward model as a reward function and further fine-tune the SFT model using the Proximal Policy Optimization (PPO) algorithm. The goal is to optimize the model's policy to maximize the reward, effectively steering it towards generating outputs that humans prefer.

The resulting models, called **InstructGPT**, were found to be significantly preferred by human labelers over the much larger GPT-3 model, despite having 100x fewer parameters. This highlights the effectiveness of alignment techniques over pure scaling.

### 1.2 Prerequisite Knowledge

#### Mathematical Concepts
- **Probability & Statistics**: Understanding of probability distributions, expectations, and log-probabilities.
- **Information Theory**: Kullback-Leibler (KL) Divergence for measuring the difference between two probability distributions.
- **Optimization**: Gradient Descent and its variants for training neural networks.
- **Calculus**: Basic understanding of derivatives for optimization.

#### Machine Learning / Computer Science Concepts
- **Large Language Models (LLMs)**: Familiarity with the Transformer architecture and how models like GPT-3 work (autoregressive text generation).
- **Supervised Learning**: Understanding of fine-tuning a pre-trained model on a labeled dataset.
- **Reinforcement Learning (RL)**: Core concepts including *policy*, *reward*, *agent*, *environment*, and *value function*.
- **Proximal Policy Optimization (PPO)**: High-level understanding of what PPO is and why it's used for stabilizing RL training.

### 1.3 Hierarchy of Topics

1.  **Mathematical Foundations**: We'll start by implementing and visualizing the key mathematical components: the pairwise ranking loss for the reward model and the PPO objective function with KL divergence.
2.  **Prerequisite Algorithms**: We'll set up our base language model, which will serve as a proxy for GPT-3.
3.  **Core Research Content: The 3-Step InstructGPT Pipeline**: We will implement a simplified, end-to-end version of the pipeline:
    - Step 1: Supervised Fine-Tuning
    - Step 2: Training a Reward Model
    - Step 3: RL Fine-Tuning with PPO
4.  **Experimental Analysis**: We will compare the outputs of the models at each stage and analyze the effect of key hyperparameters, such as the KL penalty.
5.  **Research Context & Extensions**: We'll discuss the paper's impact and the limitations and future work they identified.

### 1.4 Learning Objectives

- **Understand** the complete Reinforcement Learning from Human Feedback (RLHF) pipeline.
- **Implement** each component: SFT, Reward Modeling, and RL-based optimization.
- **Analyze** the role of the reward model and the KL penalty in aligning LLMs.
- **Evaluate** the trade-offs between model alignment and performance on standard NLP tasks (the "alignment tax").

**Estimated Time**: 2-3 hours

In [ ]:
# Installation of required libraries
!pip install transformers torch numpy pandas matplotlib ipywidgets
!pip install trl

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AutoModelForSequenceClassification, Trainer, TrainingArguments
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
import torch.nn.functional as F
import ipywidgets as widgets
from IPython.display import display, clear_output

# Set a seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Section 2: Mathematical Foundations

Before building the full pipeline, let's understand the core mathematical components that drive the training process.

### 2.1 Reward Model Loss Function

The reward model (RM) is trained on a dataset of human comparisons. For a given prompt `x`, the model generates two responses, `y_w` (the winning, preferred response) and `y_l` (the losing, rejected response). The RM, `r_θ`, learns to assign a higher scalar score to `y_w` than to `y_l`.

The loss function is a pairwise ranking loss:
$$ \text{loss}(\theta) = -\mathbb{E}_{(x, y_w, y_l) \sim D} [\log(\sigma(r_\theta(x, y_w) - r_\theta(x, y_l)))] $$

Where:
- $D$ is the dataset of human comparisons.
- $\sigma$ is the sigmoid function.
- $r_\theta(x, y)$ is the scalar score from the reward model.

This is equivalent to the binary cross-entropy loss. We want to maximize the probability that the score of the winning response is higher than the losing one, which means we want to maximize the difference $r_\theta(y_w) - r_\theta(y_l)$.

In [ ]:
def educational_reward_loss(winning_score, losing_score):
    """
    Clear implementation of the pairwise ranking loss for understanding.
    - Based directly on the formula from the InstructGPT paper.
    - Extensive comments explaining each step.
    """
    # Calculate the difference in scores
    score_diff = winning_score - losing_score
    
    # Apply the sigmoid function to convert the difference to a probability
    prob = torch.sigmoid(score_diff)
    
    # Calculate the log of the probability
    log_prob = torch.log(prob)
    
    # The loss is the negative log-likelihood (we want to maximize log_prob, so we minimize -log_prob)
    loss = -log_prob.mean()
    
    return loss

def optimized_reward_loss(winning_score, losing_score):
    """
    Efficient implementation using PyTorch's built-in functions.
    This is equivalent to Binary Cross-Entropy with Logits.
    """
    # The loss function wants to maximize the margin between the winning and losing scores.
    # This is equivalent to a binary classification problem where the target is always 1
    # (i.e., we want the winning score to be classified as 'better').
    return -F.logsigmoid(winning_score - losing_score).mean()

# --- Let's test it ---
winning_scores = torch.tensor([2.5, 0.5, -1.0])
losing_scores = torch.tensor([1.0, 0.0, -1.5])

edu_loss = educational_reward_loss(winning_scores, losing_scores)
opt_loss = optimized_reward_loss(winning_scores, losing_scores)

print(f"Educational Loss Implementation: {edu_loss.item():.4f}")
print(f"Optimized Loss Implementation:   {opt_loss.item():.4f}")
assert torch.isclose(edu_loss, opt_loss)

### 2.2 KL Divergence

The Kullback-Leibler (KL) divergence is a measure of how one probability distribution, P, is different from a second, reference probability distribution, Q. In our case, it's used to prevent the RL-tuned policy ($"\pi^{RL}"$) from straying too far from the initial SFT policy ($"\pi^{SFT}"$). This acts as a regularizer.

$$D_{KL}(P || Q) = \sum_{x \in \mathcal{X}} P(x) \log\left(\frac{P(x)}{Q(x)}\right)$$

In the context of language models, this is calculated on the token-level probability distributions.

Let's visualize it.

In [ ]:
def educational_kl_divergence(p, q):
    """
    From-scratch implementation of KL Divergence.
    Args:
        p, q: PyTorch tensors representing probability distributions.
    """
    # Ensure distributions sum to 1 and add a small epsilon for numerical stability
    epsilon = 1e-10
    p = p / p.sum()
    q = q / q.sum()
    return (p * (torch.log(p + epsilon) - torch.log(q + epsilon))).sum()

# --- Interactive Visualization of KL Divergence ---
x_axis = np.arange(10)
p_dist = F.softmax(torch.randn(10), dim=-1).numpy()

@widgets.interact(mean=widgets.FloatSlider(value=5.0, min=0.0, max=10.0, step=0.5, description='Q Mean'),
                   std=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Q Std'))
def interactive_kl_explorer(mean, std):
    # Create a normal distribution Q
    q_dist = torch.exp(-((torch.from_numpy(x_axis) - mean)**2) / (2 * std**2))
    q_dist = F.softmax(q_dist, dim=-1).numpy()
    
    kl_div = educational_kl_divergence(torch.from_numpy(p_dist), torch.from_numpy(q_dist))

    plt.figure(figsize=(10, 5))
    plt.bar(x_axis - 0.2, p_dist, width=0.4, label='P (Reference Dist)', color='blue', alpha=0.7)
    plt.bar(x_axis + 0.2, q_dist, width=0.4, label='Q (Approximating Dist)', color='red', alpha=0.7)
    plt.title(f"KL Divergence D_KL(P || Q) = {kl_div.item():.4f}")
    plt.xlabel("Event")
    plt.ylabel("Probability")
    plt.legend()
    plt.ylim(0, 1)
    plt.show()

### 2.3 PPO Objective Function

The core of the RL step is optimizing the policy. The paper uses PPO with a specific objective function that balances the reward from the RM with the KL penalty.

The objective for the PPO-ptx model (which includes a pre-training gradient mix) is:
$$ \text{objective}(\phi) = \mathbb{E}_{(x,y) \sim D_{\pi_{\phi}^{RL}}} [r_\theta(x, y) - \beta \log(\pi_{\phi}^{RL}(y|x) / \pi^{SFT}(y|x))] + \gamma \mathbb{E}_{x \sim D_{pretrain}} [\log(\pi_{\phi}^{RL}(x))] $$ 

Let's break this down:
1.  **Reward Term**: $\mathbb{E}_{(x,y) \sim D_{\pi_{\phi}^{RL}}} [r_\theta(x, y)]$
    - This is the expected reward from the reward model for generations `y` from our current RL policy `π_φ^RL` given prompts `x`. This pushes the model to generate text that the RM scores highly.

2.  **KL Penalty Term**: $- \beta \log(\pi_{\phi}^{RL}(y|x) / \pi^{SFT}(y|x))$
    - This is the KL divergence between the RL policy and the initial SFT policy. The term $\log(\pi_{\phi}^{RL} / \pi^{SFT})$ is an estimate of the per-token KL divergence. Multiplying by a coefficient `β` controls the strength of this penalty. A higher `β` keeps the model from deviating too much from its initial, safer behavior.

3.  **Pre-training Mix Term**: $+ \gamma \mathbb{E}_{x \sim D_{pretrain}} [\log(\pi_{\phi}^{RL}(x))]$
    - This term encourages the model to maintain its performance on the original pre-training distribution, helping to mitigate the "alignment tax" where the model loses general capabilities.

In our simplified implementation, we will focus on the first two terms, as they are the core of the RLHF process.

## Section 3: Prerequisite Algorithms - Setting up the Base Model

The paper uses versions of the massive GPT-3 model. For this notebook, we need a smaller, accessible, pre-trained causal language model. We will use **GPT-2** from Hugging Face as our stand-in base model. All the principles and techniques apply, but on a smaller scale.

In [ ]:
# Load a pre-trained model and tokenizer
model_name = 'distilgpt2'
base_model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# Set padding token if it's not set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    base_model.config.pad_token_id = base_model.config.eos_token_id

print(f"Loaded base model: {model_name}")
print("Let's see a sample generation from the base model before any fine-tuning:")

prompt = "Write a short story about a robot who discovers music."
inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True)
output_sequences = base_model.generate(input_ids=inputs['input_ids'], max_length=100)

print(tokenizer.decode(output_sequences[0], skip_special_tokens=True))

As you can see, the base model understands the prompt to some extent but its completion is often repetitive, generic, and doesn't follow the instruction very well. Our goal is to improve this through alignment.

## Section 4: Core Research Content - The 3-Step InstructGPT Pipeline

We will now implement a simplified version of the entire pipeline.

### Step 1: Supervised Fine-Tuning (SFT)

First, we fine-tune our base model on a small dataset of high-quality instruction-response pairs. This teaches the model the general format of following instructions.

Let's create a synthetic dataset for this purpose.

In [ ]:
# Create a small, synthetic SFT dataset
sft_data = [
    {"prompt": "Summarize the following text: The sun is a star at the center of the Solar System. It is a nearly perfect sphere of hot plasma.", "response": "The sun is a star made of hot plasma at the center of our solar system."},
    {"prompt": "Translate to French: 'Hello, how are you?'", "response": "'Bonjour, comment ça va?'"},
    {"prompt": "Write a tagline for a coffee shop.", "response": "Your daily grind, perfected."},
    {"prompt": "Explain gravity in one sentence.", "response": "Gravity is the force by which a planet or other body draws objects toward its center."}
]

sft_dataset_df = pd.DataFrame(sft_data)
print("SFT Dataset:")
display(sft_dataset_df)

# Format the data for training
formatted_sft_data = []
for _, row in sft_dataset_df.iterrows():
    formatted_sft_data.append(f"Prompt: {row['prompt']}\nResponse: {row['response']}")

# For simplicity in this notebook, we'll use a very basic training loop.
# In a real-world scenario, you'd use Hugging Face's Trainer or a custom PyTorch loop.

# Let's pretend we fine-tuned. In a real scenario, this step takes time.
# We'll load a fresh copy of the model to represent the SFT model to avoid altering the base model.
sft_model = GPT2LMHeadModel.from_pretrained(model_name)
sft_model.config.pad_token_id = sft_model.config.eos_token_id

print("\n--- SFT Model is ready (simulated fine-tuning) ---")
print("In a full implementation, you would fine-tune the model on the SFT dataset here.")

# To actually train, you would do something like this:
# class SFTDataset(torch.utils.data.Dataset):
#     def __init__(self, encodings):
#         self.encodings = encodings
#     def __getitem__(self, idx):
#         return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
#     def __len__(self):
#         return len(self.encodings.input_ids)

# tokenized_data = tokenizer(formatted_sft_data, padding=True, truncation=True, return_tensors='pt')
# sft_dataset = SFTDataset(tokenized_data)
# training_args = TrainingArguments(output_dir='./sft_model', num_train_epochs=1, per_device_train_batch_size=1)
# trainer = Trainer(model=sft_model, args=training_args, train_dataset=sft_dataset)
# trainer.train()


### Step 2: Reward Modeling (RM)

Now, we take the SFT model and generate several responses to a new set of prompts. We then create a synthetic preference dataset, simulating a human labeler choosing the 'better' response. This dataset will be used to train our reward model.

The reward model itself has the same architecture as the SFT model, but with its language modeling head replaced by a single linear layer that outputs a scalar value (the reward).

In [ ]:
# 1. Create prompts for generating comparison data
rm_prompts = [
    "What is the capital of Japan?",
    "Write a haiku about winter."
]

# 2. Generate responses from the SFT model (we'll create them manually for demonstration)
comparison_data = [
    {
        "prompt": rm_prompts[0],
        "chosen": "The capital of Japan is Tokyo.", # Good, factual answer
        "rejected": "Japan is an island country in East Asia." # True, but doesn't answer the question
    },
    {
        "prompt": rm_prompts[1],
        "chosen": "White snow falls softly,\nWinter's breath, a silent hush,\nWorld sleeps under ice.", # Good haiku
        "rejected": "Winter is very cold." # Bad, not a haiku
    }
]

rm_dataset_df = pd.DataFrame(comparison_data)
print("Reward Model Preference Dataset:")
display(rm_dataset_df)

# 3. Set up the Reward Model
# We use AutoModelForSequenceClassification with 1 label to output a scalar score
reward_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)
reward_model.config.pad_token_id = reward_model.config.eos_token_id

print("\n--- Reward Model is ready to be trained ---")

# In a real scenario, we would tokenize the chosen/rejected pairs and train the RM
# using the `optimized_reward_loss` we defined earlier.
class RewardDataset(torch.utils.data.Dataset):
    def __init__(self, tokenizer, data):
        self.tokenizer = tokenizer
        self.data = data
        self.max_len = 128

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        prompt = item['prompt']
        chosen = item['chosen']
        rejected = item['rejected']
        
        chosen_tokens = self.tokenizer(prompt + " " + chosen, max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        rejected_tokens = self.tokenizer(prompt + " " + rejected, max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        
        return {
            "input_ids_chosen": chosen_tokens["input_ids"].squeeze(0),
            "attention_mask_chosen": chosen_tokens["attention_mask"].squeeze(0),
            "input_ids_rejected": rejected_tokens["input_ids"].squeeze(0),
            "attention_mask_rejected": rejected_tokens["attention_mask"].squeeze(0),
        }

class RewardTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        rewards_chosen = model(input_ids=inputs["input_ids_chosen"], attention_mask=inputs["attention_mask_chosen"])[0]
        rewards_rejected = model(input_ids=inputs["input_ids_rejected"], attention_mask=inputs["attention_mask_rejected"])[0]
        loss = optimized_reward_loss(rewards_chosen, rewards_rejected)
        return (loss, {"rewards_chosen": rewards_chosen, "rewards_rejected": rewards_rejected}) if return_outputs else loss


print("Training Reward Model...")
rm_dataset = RewardDataset(tokenizer, comparison_data)
training_args = TrainingArguments(
    output_dir="./rm_model",
    num_train_epochs=5, # Train for more epochs on this tiny dataset
    per_device_train_batch_size=1,
    logging_steps=1,
)

rm_trainer = RewardTrainer(
    model=reward_model,
    args=training_args,
    train_dataset=rm_dataset,
)
rm_trainer.train()
print("Reward Model training complete.")

# Let's test the trained reward model
text_good = "What is the capital of Japan? The capital of Japan is Tokyo."
text_bad = "What is the capital of Japan? Japan is an island country in East Asia."

good_tokens = tokenizer(text_good, return_tensors='pt')
bad_tokens = tokenizer(text_bad, return_tensors='pt')

reward_model.eval()
with torch.no_grad():
    good_reward = reward_model(**good_tokens).logits.item()
    bad_reward = reward_model(**bad_tokens).logits.item()

print(f'\nReward for good answer: {good_reward:.2f}')
print(f'Reward for bad answer:  {bad_reward:.2f}')
assert good_reward > bad_reward

The reward model successfully learned to assign a higher score to the preferred response.

### Step 3: RL Fine-Tuning with PPO

This is the final and most complex step. We use the trained Reward Model from Step 2 to provide a reward signal to the SFT model from Step 1. The PPO algorithm optimizes the SFT model (the policy) to generate responses that maximize the reward score, while the KL-divergence term keeps it from straying too far from its original instruction-following capabilities.

We will use the `trl` library from Hugging Face, which provides a convenient `PPOTrainer` for this exact purpose.

In [ ]:
# 1. Set up PPO configuration
ppo_config = PPOConfig(
    model_name=model_name,
    learning_rate=1.41e-5,
    batch_size=2, # Use the full dataset
    mini_batch_size=1,
    log_with="tensorboard",
    project_kwargs={"logging_dir": "./ppo_logs"}
)

# 2. Create the PPO Trainer
# The PPOTrainer needs the SFT model (as the policy to be trained) 
# and the Reward Model to score the generations.
# TRL's AutoModelForCausalLMWithValueHead combines the actor and critic.

ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(model_name)
ppo_model.config.pad_token_id = ppo_model.config.eos_token_id

# The PPOTrainer takes the SFT model as a reference for the KL penalty
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=ppo_model,
    ref_model=None, # TRL will create a reference model from the ppo_model
    tokenizer=tokenizer,
    dataset=None # We will provide data manually
)

# 3. Create RL prompts and tokenize them
rl_prompts = [
    "Explain the concept of photosynthesis simply.",
    "List three benefits of regular exercise."
]
query_tensors = [tokenizer.encode(p, return_tensors="pt").squeeze(0) for p in rl_prompts]

print("--- Starting PPO Training Loop ---")

# 4. The PPO Training Loop
generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
    "max_new_tokens": 32,
}

for epoch in range(2): # 2 epochs
    for i, query_tensor in enumerate(query_tensors):
        # Generate a response from the policy (SFT model)
        response_tensor = ppo_trainer.generate(query_tensor, **generation_kwargs)
        response_txt = tokenizer.decode(response_tensor.squeeze())
        
        # Get the reward from the reward model
        # The full text (prompt + response) is passed to the RM
        full_txt = rl_prompts[i] + response_txt
        rm_inputs = tokenizer(full_txt, return_tensors='pt')
        reward = reward_model(**rm_inputs).logits.detach()
        
        # The PPO step performs the optimization
        stats = ppo_trainer.step([query_tensor], [response_tensor], [reward])
        ppo_trainer.log_stats(stats, {"prompt": rl_prompts[i]}, reward.squeeze())

print("\n--- PPO Training Finished ---")

## Section 5: Experimental Analysis

Now that we have gone through the full pipeline, let's compare the outputs from the three models: the base model, the SFT model, and the final PPO-tuned model.

In [ ]:
def generate_text(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors='pt')
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0], skip_special_tokens=True)

test_prompt = "Write a short, positive sentence about learning."

# In this simplified notebook, the sft_model wasn't actually trained, so it's the same as the base model.
# The ppo_model.model is the one that was updated.
final_ppo_model = ppo_model.model

base_output = generate_text(base_model, tokenizer, test_prompt)
ppo_output = generate_text(final_ppo_model, tokenizer, test_prompt)

print(f"--- Comparison for prompt: '{test_prompt}' ---")
print("\n[Base Model Output]")
print(base_output)

print("\n[Final PPO Model Output]")
print(ppo_output)


While the results on this tiny scale are illustrative rather than definitive, the goal of the PPO model is to be more aligned with what the reward model considers 'good'. In a full-scale training, the PPO output would be noticeably more helpful and on-topic.

### 5.1 Parameter Sensitivity: The KL Coefficient (β)

The KL coefficient `β` is a critical hyperparameter. It controls the trade-off between maximizing the reward and staying close to the original SFT policy.
- **Low `β`**: The model is free to chase high rewards, which can lead to *reward hacking*—generating nonsensical text that happens to fool the reward model. The output may lose coherence and deviate significantly from natural language.
- **High `β`**: The model is heavily penalized for deviating from the SFT policy. The RL training has little effect, and the model's output will be very similar to the SFT model's output.

Let's visualize this trade-off.

In [ ]:
def kl_beta_effect_explorer():
    reward = np.linspace(-3, 3, 100)
    kl = np.linspace(0, 5, 100)
    
    @widgets.interact(beta=widgets.FloatSlider(value=0.2, min=0.0, max=2.0, step=0.05, description='Beta (β)'))
    def plot_objective(beta):
        # Objective = Reward - beta * KL
        objective = reward - beta * kl
        
        plt.figure(figsize=(8, 6))
        plt.plot(reward, objective, label=f'Objective with β={beta:.2f}')
        plt.title('Effect of KL Coefficient (β) on the RL Objective')
        plt.xlabel('Reward from RM')
        plt.ylabel('Final Objective Value (Reward - β * KL)')
        plt.grid(True)
        plt.legend()
        plt.show()
        
        print("Interpretation:")
        print("This plot shows the final objective value for a fixed KL divergence of `kl` (x-axis is the reward). ")
        if beta < 0.1:
            print("With a very low beta, the objective is almost entirely driven by the reward. The model will prioritize high-reward outputs, even if they are very different from the SFT model (high KL).")
        elif beta > 1.0:
            print("With a high beta, the KL penalty is severe. Even a high reward can result in a low objective if the output deviates from the SFT model. The model will be conservative and stay close to the SFT policy.")
        else:
            print("A moderate beta balances the reward and the KL penalty, encouraging helpful outputs that are not radically different from the initial fine-tuned model.")

kl_beta_effect_explorer()

### 5.2 The "Alignment Tax"

A key finding of the paper is that this alignment process can sometimes degrade performance on certain public NLP datasets (e.g., SQuAD, DROP). This performance drop is called the **alignment tax**. The model becomes better at being helpful and harmless in a conversational context but might become slightly worse at specific, academic NLP tasks.

The authors mitigate this by mixing in gradients from the original pre-training dataset during the PPO phase (the `γ` term in the objective function). This reminds the model of its foundational capabilities and reduces the alignment tax.

## Section 6: Research Context & Extensions


### 6.1 Research Contribution in Context

The InstructGPT paper was a landmark publication that provided a concrete, scalable, and effective recipe for aligning language models with human intent. It built upon previous work in areas like summarization with human feedback but applied the technique to a much broader range of tasks.

This work laid the public foundation for the technology behind **ChatGPT**. While ChatGPT is a more advanced and larger-scale application, it follows the same core principles of SFT, Reward Modeling, and RLHF detailed in this paper.

The key takeaway is that explicit alignment with human preferences is a more direct and efficient path to creating helpful AI assistants than simply scaling up model size.

### 6.2 Limitations and Future Directions

The authors are candid about the limitations of their work, which point to important areas of ongoing research:

- **Who are we aligning to?**: The model's behavior is a direct reflection of the values and preferences of the human labelers and the instructions they were given. This raises critical questions about fairness, representation, and whose values should be embedded in AI systems. The labelers were a small group of contractors, not a representation of all of humanity.
- **Instruction Following Failures**: InstructGPT still makes simple mistakes. It can fail to follow complex constraints, assume false premises in a prompt are true, or be overly hedging.
- **Harmful Outputs**: The model can still be prompted to generate toxic, biased, or otherwise harmful content. The alignment is not perfect and can be overcome by adversarial prompts.
- **Improving Truthfulness**: While better than GPT-3, InstructGPT still hallucinates or makes up facts. Future work like WebGPT (which allows the model to browse the web) aims to address this directly.

This concludes our interactive exploration of the InstructGPT paper. We have implemented the core components of the RLHF pipeline and analyzed the key concepts that make it an effective technique for aligning language models.